In [ ]:
import re
import pandas as pd

def merge_bed_files(file1, file2, file3, file4, file5, file6, output_file=None):
    """
    Merge six BED-like files that have matching chromosome, start, and end coordinates.
    Each file's lines are expected to look like:
        chr10_MATERNAL    [42102611, 42139864]    0.005911728253420023    50.29890210184415

    Columns:
       1) chromosome (e.g., 'chr10_MATERNAL')
       2) bracketed [start, end]
       3) a numeric "density" (or other) value
       4) another numeric (or any) value

    We collect only the 3rd column (the "density") from each file as
    value1..value6 in the merged output.
    
    We also compute a new column 'island_length' = end - start.

    The output DataFrame has columns:
        [chromosome, start, end, island_length, 
         old passed CENPA, young passed CENPA, old passed mC, young passed mC, 
         H3K9me3_young, H3K9me3_old]

    Parameters
    ----------
    file1, file2, file3, file4, file5, file6 : str
        Paths to the six input files.
    output_file : str, optional
        If provided, the merged DataFrame will be written to this file (tab-delimited).

    Returns
    -------
    df : pandas.DataFrame
        Merged DataFrame with the columns listed above.
    """

    # Regex to match lines of the form:
    #   chromosome  [start, end]  col3  col4
    #
    # Allows for variable whitespace and optional spaces within the brackets.
    line_regex = re.compile(
        r'^(\S+)\s+\[\s*(\d+)\s*,\s*(\d+)\s*\]\s+(\S+)\s+(\S+)$'
    )

    def parse_line(line):
        """
        Parse a single line using the above regex, returning:
           chromosome, start, end, col3
        (col4 is captured but ignored in this example.)
        """
        line = line.strip()
        match = line_regex.match(line)
        if not match:
            raise ValueError(f"Line does not match format:\n{line}")
        chrom, start, end, col3, col4 = match.groups()
        return chrom, start, end, col3

    merged_data = []

    # Read each file in parallel line by line
    with open(file1, "r") as f1, open(file2, "r") as f2, \
         open(file3, "r") as f3, open(file4, "r") as f4, \
         open(file5, "r") as f5, open(file6, "r") as f6:
        
        for l1, l2, l3, l4, l5, l6 in zip(f1, f2, f3, f4, f5, f6):
            # Parse lines from each file
            c1, s1, e1, v1 = parse_line(l1)
            c2, s2, e2, v2 = parse_line(l2)
            c3, s3, e3, v3 = parse_line(l3)
            c4, s4, e4, v4 = parse_line(l4)
            c5, s5, e5, v5 = parse_line(l5)
            c6, s6, e6, v6 = parse_line(l6)
            
            # Optional: Check consistency across files.
            # Uncomment the following block if you want to enforce that all files have matching coordinates.
            # if not (c1 == c2 == c3 == c4 == c5 == c6 and s1 == s2 == s3 == s4 == s5 == s6 and e1 == e2 == e3 == e4 == e5 == e6):
            #     raise ValueError(
            #         f"Mismatch between lines:\n{l1}{l2}{l3}{l4}{l5}{l6}"
            #     )

            # Convert start/end to integers to compute the island length
            s_int = int(s1)
            e_int = int(e1)
            island_length = e_int - s_int

            # Append a row with the new column 'island_length' and tracks from all six files
            merged_data.append([
                c1, s_int, e_int, island_length,
                v1, v2, v3, v4,  # from first four files
                v5, v6         # from the new two files (H3K9me3 tracks)
            ])

    # Build DataFrame with appropriate column names
    df = pd.DataFrame(
        merged_data,
        columns=[
            "chromosome", "start", "end", "island_length",
            "old passed CENPA", "young passed CENPA", "old passed mC", "young passed mC",
            "H3K9me3_young", "H3K9me3_old"
        ]
    )

    # If an output file is specified, write a tab-delimited file
    if output_file:
        df.to_csv(output_file, sep="\t", index=False)

    return df


In [ ]:

sub_island_df_merged = merge_bed_files(
    "/private/groups/migalab/dan/data_analysis/young_old_analysis/cenpa_old_subisland_region_density_scores_A.csv",
    "/private/groups/migalab/dan/data_analysis/young_old_analysis/cenpa_young_subisland_region_density_scores_A.csv",
    "/private/groups/migalab/dan/data_analysis/young_old_analysis/mCpG_old_subisland_region_density_scores_CG.csv",
    "/private/groups/migalab/dan/data_analysis/young_old_analysis/mCpG_young_baseline_region_density_scores_CG.csv",
    "/private/groups/migalab/dan/data_analysis/young_old_analysis/h3k9me3_young_subisland_region_density_scores_A.csv",
    "/private/groups/migalab/dan/data_analysis/young_old_analysis/h3k9me3_old_subisland_region_density_scores_A.csv",
    
    output_file= "/private/groups/migalab/dan/data_analysis/young_old_analysis/old_passaged_islands_comparison/young_old_islands_merged.csv"
)

non_island_df_merged = merge_bed_files(
    "/private/groups/migalab/dan/data_analysis/young_old_analysis/cenpa_old_non_subisland_region_density_scores_A.csv",
    "/private/groups/migalab/dan/data_analysis/young_old_analysis/cenpa_young_non_subisland_region_density_scores_A.csv",
    "/private/groups/migalab/dan/data_analysis/young_old_analysis/mCpG_old_non_subisland_region_density_scores_CG.csv",
    "/private/groups/migalab/dan/data_analysis/young_old_analysis/mCpG_young_non_subisland_region_density_scores_CG.csv",
    "/private/groups/migalab/dan/data_analysis/young_old_analysis/h3k9me3_young_non_subisland_region_density_scores_A.csv",
    "/private/groups/migalab/dan/data_analysis/young_old_analysis/h3k9me3_old_non_subisland_region_density_scores_A.csv",
    output_file= "/private/groups/migalab/dan/data_analysis/young_old_analysis/old_passaged_islands_comparison/young_old_non_islands_merged.csv"
)


sub_CDR_df_merged = merge_bed_files(
    "/private/groups/migalab/dan/data_analysis/young_old_analysis/CENPA_old_CDR_dict_region_density_scores_A.csv",
    "/private/groups/migalab/dan/data_analysis/young_old_analysis/CENPA_young_CDR_dict_region_density_scores_A.csv",
    "/private/groups/migalab/dan/data_analysis/young_old_analysis/mCpG_old_subCDR_region_density_scores_CG.csv",
    "/private/groups/migalab/dan/data_analysis/young_old_analysis/mCpG_young_subCDR_region_density_scores_CG.csv",
    "/private/groups/migalab/dan/data_analysis/young_old_analysis/h3k9me3_young_subCDR_region_density_scores_A.csv",
    "/private/groups/migalab/dan/data_analysis/young_old_analysis/h3k9me3_old_subCDR_region_density_scores_A.csv",
    
    output_file= "/private/groups/migalab/dan/data_analysis/young_old_analysis/old_passaged_islands_comparison/young_old_CDR_merged.csv"
)

In [ ]:
mCpG_comparison_merged = merge_bed_files(
    "/private/groups/migalab/dan/data_analysis/young_old_analysis/h3k9_old_chrom_region_density_scores_CG.csv",
    "/private/groups/migalab/dan/data_analysis/young_old_analysis/h3k9_mCpG_young_chrom_region_density_scores_CG.csv",
    "/private/groups/migalab/dan/data_analysis/young_old_analysis/mCpG_old_subCDR_region_density_scores_CG.csv",
    "/private/groups/migalab/dan/data_analysis/young_old_analysis/mCpG_young_subCDR_region_density_scores_CG.csv",
    "/private/groups/migalab/dan/data_analysis/young_old_analysis/mCpG_young_non_CDR_region_density_scores_CG.csv",
    "/private/groups/migalab/dan/data_analysis/young_old_analysis/mCpG_old_non_CDR_region_density_scores_CG.csv",
    
    output_file= "/private/groups/migalab/dan/data_analysis/young_old_analysis/old_passaged_islands_comparison/young_old_CDR_merged.csv"
)

In [ ]:
def filter_df_by_bed(df, bed_file):
    """
    Filter a DataFrame so that it only keeps rows that overlap any interval in a given BED file.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame produced by merge_bed_files, with columns at least:
            ['chromosome', 'start', 'end', 'island_length', ...]
    bed_file : str
        Path to a BED file (or similar) where:
           - The 1st column is the chromosome
           - The 2nd column is the start
           - The 3rd column is the end
           - The rest can be ignored

    Returns
    -------
    filtered_df : pd.DataFrame
        DataFrame that includes only the rows whose interval (chromosome, start, end)
        overlaps any interval specified in the provided BED file.
    """
    # Build a dictionary mapping chromosome -> list of (start, end) intervals from the BED file.
    bed_intervals = {}
    with open(bed_file, "r") as bf:
        for line in bf:
            line = line.strip()
            if not line:
                continue  # Skip empty lines.
            parts = line.split()
            if len(parts) < 3:
                continue  # Skip lines that don't have at least 3 columns.
            bed_chrom = parts[0]
            bed_start = int(parts[1])
            bed_end   = int(parts[2])
            if bed_chrom not in bed_intervals:
                bed_intervals[bed_chrom] = []
            bed_intervals[bed_chrom].append((bed_start, bed_end))

    
    # Define a helper function to check for overlap between two intervals.
    # Using inclusive comparisons (>= and <=) so that intervals that touch or partially overlap count.
    def intervals_overlap(start1, end1, start2, end2):
        return end1 > start2 and start1 < end2

    # Filter df: for each row, check if its interval overlaps any interval in the BED intervals for that chromosome.
    mask = df.apply(
        lambda row: (row["chromosome"] in bed_intervals and 
                     any(intervals_overlap(row["start"], row["end"], bed_start, bed_end)
                         for (bed_start, bed_end) in bed_intervals[row["chromosome"]])
                    ),
        axis=1
    )

    def debug_overlap(row):
        if row["chromosome"] not in bed_intervals:
            print("Chromosome not in bed_intervals:", row["chromosome"])
            return False
        for bed_start, bed_end in bed_intervals[row["chromosome"]]:
            if intervals_overlap(int(row["start"]), int(row["end"]), bed_start, bed_end):
                return True
        print("No overlap found for:", row["chromosome"], row["start"], row["end"])
        return False

    mask = df.apply(debug_overlap, axis=1)

    #print (mask)
    filtered_df = df[mask].copy()  # Create a new DataFrame from the filtered rows.
    return filtered_df


In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


# Define regex for parsing a baseline BED file line.
# Expected line format: chromosome  [start, end]  density  col4
line_regex = re.compile(r'^(\S+)\s+\[\s*(\d+)\s*,\s*(\d+)\s*\]\s+(\S+)\s+(\S+)$')

def load_baseline(baseline_file):
    """
    Load a baseline BED file and return a dictionary mapping each chromosome to its baseline density.
    
    Parameters
    ----------
    baseline_file : str
        Path to the baseline BED file.
    
    Returns
    -------
    dict
        Dictionary with keys as chromosome names and values as baseline density (float).
    """
    baseline_dict = {}
    with open(baseline_file, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            match = line_regex.match(line)
            if not match:
                raise ValueError(f"Line does not match expected format:\n{line}")
            chrom, start, end, density, _ = match.groups()
            baseline_dict[chrom] = float(density)
    return baseline_dict

baseline_file_old_passed_CENPA   = "/private/groups/migalab/dan/data_analysis/young_old_analysis/cenpa_old_baseline_dict_region_density_scores_A.csv"
baseline_file_young_passed_CENPA = "/private/groups/migalab/dan/data_analysis/young_old_analysis/cenpa_young_baseline_dict_region_density_scores_A.csv"
baseline_file_H3K9me3_young      = "/private/groups/migalab/dan/data_analysis/young_old_analysis/h3k9me3_young_baseline_region_density_scores_A.csv"
baseline_file_H3K9me3_old        = "/private/groups/migalab/dan/data_analysis/young_old_analysis/h3k9me3_old_baseline_region_density_scores_A.csv"

baseline_file_mCpG_young = "/private/groups/migalab/dan/data_analysis/young_old_analysis/mCpG_young_baseline_region_density_scores_CG.csv"
baseline_file_mCpG_old = "/private/groups/migalab/dan/data_analysis/young_old_analysis/mCpG_old_baseline_region_density_scores_CG.csv"


In [ ]:
baseline_old_passed_CENPA   = load_baseline(baseline_file_old_passed_CENPA)
baseline_young_passed_CENPA = load_baseline(baseline_file_young_passed_CENPA)
baseline_H3K9me3_young      = load_baseline(baseline_file_H3K9me3_young)
baseline_H3K9me3_old        = load_baseline(baseline_file_H3K9me3_old)
baseline_mCpG_young         = load_baseline(baseline_file_mCpG_young)
baseline_mCpG_old           = load_baseline(baseline_file_mCpG_old)

In [ ]:
def plot_log_fold_changes(df, 
                          baseline_old_passed_CENPA, baseline_young_passed_CENPA,
                          baseline_H3K9me3_young, baseline_H3K9me3_old,
                          baseline_mCpG_young, baseline_mCpG_old,
                          output_path=None):
    """
    Calculate fold changes and log2 fold changes relative to provided baseline dictionaries,
    then create a box plot with individual data points overlaid.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing at least the following columns:
          - "chromosome"
          - "old passed CENPA"
          - "young passed CENPA"
          - "H3K9me3_young"
          - "H3K9me3_old"
          - "young passed mC"
          - "old passed mC"
    baseline_old_passed_CENPA, baseline_young_passed_CENPA,
    baseline_H3K9me3_young, baseline_H3K9me3_old,
    baseline_mCpG_young, baseline_mCpG_old : dict
        Dictionaries with chromosome-specific baseline values.
    output_path : str, optional
        If provided, the plot will be saved to this path.
    """

    # Nested function to calculate the fold change for a given row.
    def calculate_fold_change(row, column_name, baseline_dict):
        chrom = row["chromosome"]
        try:
            value = float(row[column_name])
        except ValueError:
            return None
        baseline = baseline_dict.get(chrom)
        if baseline is None or baseline == 0:
            return None
        return value / baseline

    # Calculate fold changes for each measurement.
    df["old passed CENPA fold change"] = df.apply(
        lambda row: calculate_fold_change(row, "old passed CENPA", baseline_old_passed_CENPA), axis=1)
    df["young passed CENPA fold change"] = df.apply(
        lambda row: calculate_fold_change(row, "young passed CENPA", baseline_young_passed_CENPA), axis=1)
    df["H3K9me3_young fold change"] = df.apply(
        lambda row: calculate_fold_change(row, "H3K9me3_young", baseline_H3K9me3_young), axis=1)
    df["H3K9me3_old fold change"] = df.apply(
        lambda row: calculate_fold_change(row, "H3K9me3_old", baseline_H3K9me3_old), axis=1)
    df["mCpG_young fold change"] = df.apply(
        lambda row: calculate_fold_change(row, "young passed mC", baseline_mCpG_young), axis=1)
    df["mCpG_old fold change"] = df.apply(
        lambda row: calculate_fold_change(row, "old passed mC", baseline_mCpG_old), axis=1)

    # Compute log2 fold changes (only for positive values).
    df["old passed CENPA log fold change"] = df["old passed CENPA fold change"].apply(
        lambda x: np.log2(x) if x is not None and x > 0 else None)
    df["young passed CENPA log fold change"] = df["young passed CENPA fold change"].apply(
        lambda x: np.log2(x) if x is not None and x > 0 else None)
    df["H3K9me3_young log fold change"] = df["H3K9me3_young fold change"].apply(
        lambda x: np.log2(x) if x is not None and x > 0 else None)
    df["H3K9me3_old log fold change"] = df["H3K9me3_old fold change"].apply(
        lambda x: np.log2(x) if x is not None and x > 0 else None)
    df["mCpG_young log fold change"] = df["mCpG_young fold change"].apply(
        lambda x: np.log2(x) if x is not None and x > 0 else None)
    df["mCpG_old log fold change"] = df["mCpG_old fold change"].apply(
        lambda x: np.log2(x) if x is not None and x > 0 else None)

    # Prepare data for the box plot (the order matters for custom colors and labels).
    log_fold_change_columns = [
        "mCpG_young log fold change",
        "mCpG_old log fold change",
        "H3K9me3_young log fold change", 
        "H3K9me3_old log fold change", 
        "young passed CENPA log fold change", 
        "old passed CENPA log fold change"
    ]
    data_to_plot_log = [df[col].dropna() for col in log_fold_change_columns]

    # Create the box plot using an explicit figure and axes object
    fig, ax = plt.subplots(figsize=(10, 6))
    box = ax.boxplot(data_to_plot_log, patch_artist=True, showfliers=False, widths=0.5)

    # Set custom colors for each pair:
    # - mCpG pair (indices 0 and 1): "#B91372"
    # - H3K9me3 pair (indices 2 and 3): "#3F784C"
    # - CENPA pair (indices 4 and 5): "#56638A"
    for i, b in enumerate(box['boxes']):
        if i in [0, 1]:
            b.set_facecolor('#B91372')
        elif i in [2, 3]:
            b.set_facecolor('#3F784C')
        elif i in [4, 5]:
            b.set_facecolor('#56638A')

    # Make the median lines thicker and print their median values.
    for i, median in enumerate(box['medians']):
        median.set_linewidth(3)
        med_value = median.get_ydata()[0]
        print(f"Box {i+1} median: {med_value}")

    # Overlay individual data points with a slight jitter.
    for i, dataset in enumerate(data_to_plot_log):
        x_jitter = np.random.normal(loc=i + 1, scale=0.05, size=len(dataset))
        ax.scatter(x_jitter, dataset, alpha=0.7, color='black', s=10, zorder=2)

    # Set axis labels and title.
    ax.set_xticks(range(1, len(log_fold_change_columns) + 1))
    ax.set_xticklabels(["mCpG_young", "mCpG_old", "H3K9me3_young", "H3K9me3_old", "young passed CENPA", "old passed CENPA"])
    ax.set_ylabel("Log2 Fold Change vs Chromosome Baseline")
    ax.set_title("Log2 Fold Change of Datasets vs Chromosome Baseline")
    fig.tight_layout()

    # Save first (before show clears the figure), then display
    if output_path is not None:
        fig.savefig(output_path, format="svg" if output_path.endswith(".svg") else "png", bbox_inches="tight")
        print(f"[saved] {output_path}")

    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def plot_log_fold_changes_mC_comparison(df, output_path=None):
    """
    Calculate log2 ratios of densities between paired conditions (old vs young),
    then create a box plot with individual data points overlaid.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing at least the following columns:
          - "chromosome"
          - "old passed CENPA"
          - "young passed CENPA"
          - "H3K9me3_young"
          - "H3K9me3_old"
          - "young passed mC"
          - "old passed mC"
    output_path : str, optional
        If provided, the plot will be saved to this path.
    """

    # Nested function to calculate the log2 ratio for a given row between old and young.
    def calculate_log_ratio(row, young_col, old_col):
        try:
            young_value = float(row[young_col])
            old_value = float(row[old_col])
        except ValueError:
            return None
        if young_value == 0 or young_value <= 0 or old_value <= 0:  # Avoid division by zero or log of zero/negative
            return None
        return np.log2(old_value / young_value)  # Changed to old/young

    # Calculate log2 ratios for each paired measurement (old vs young).
    df["mCpG chrom arm log ratio (old/young)"] = df.apply(
        lambda row: calculate_log_ratio(row, "young passed CENPA", "old passed CENPA"), axis=1)
    df["mCpG non sub CDR log ratio (old/young)"] = df.apply(
        lambda row: calculate_log_ratio(row, "H3K9me3_young", "H3K9me3_old"), axis=1)
    df["mCpG sub CDR log ratio (old/young)"] = df.apply(
        lambda row: calculate_log_ratio(row, "young passed mC", "old passed mC"), axis=1)

    # Prepare data for the box plot.
    log_ratio_columns = [
        "mCpG sub CDR log ratio (old/young)",
        "mCpG non sub CDR log ratio (old/young)",
        "mCpG chrom arm log ratio (old/young)"
    ]
    data_to_plot_log = [df[col].dropna() for col in log_ratio_columns]

    # Create the box plot.
    plt.figure(figsize=(10, 6))
    box = plt.boxplot(data_to_plot_log, patch_artist=True, showfliers=False, widths=0.5)

    # Set custom colors for each measurement type:
    # - All mCpG-related (indices 0, 1, 2): "#B91372"
    for i, b in enumerate(box['boxes']):
        b.set_facecolor('#B91372')  # All boxes use the same color as in your original

    # Make the median lines thicker and print their median values.
    for i, median in enumerate(box['medians']):
        median.set_linewidth(3)
        med_value = median.get_ydata()[0]
        print(f"Box {i+1} median: {med_value}")

    # Overlay individual data points with a slight jitter.
    for i, dataset in enumerate(data_to_plot_log):
        x_jitter = np.random.normal(loc=i + 1, scale=0.05, size=len(dataset))
        plt.scatter(x_jitter, dataset, alpha=0.7, color='black', s=10, zorder=2)

    # Set axis labels and title.
    plt.xticks(range(1, len(log_ratio_columns) + 1), 
               ["mCpG sub CDR (old/young)", "mCpG non sub CDR (old/young)", "mCpG chrom arm (old/young)"])
    plt.ylabel("Log2 Ratio (Old / Young)")
    plt.title("Log2 Ratio of Densities (Old vs Young)")

    # Save the figure if an output path is provided.
    if output_path is not None:
        plt.savefig(output_path, format='tiff', dpi=800, bbox_inches='tight')
    
    plt.show()

# Example usage (replace with your actual DataFrame):
# df = pd.DataFrame({
#     "chromosome": ["chr1", "chr2", ...],
#     "old passed CENPA": [...],
#     "young passed CENPA": [...],
#     "H3K9me3_young": [...],
#     "H3K9me3_old": [...],
#     "young passed mC": [...],
#     "old passed mC": [...]
# })
#plot_log_fold_changes_mC_comparison(df, output_path="output.tiff")

In [ ]:
plot_log_fold_changes_mC_comparison(sub_island_df_merged)

In [ ]:

plot_log_fold_changes(sub_island_df_merged, 
                          baseline_old_passed_CENPA, baseline_young_passed_CENPA,
                          baseline_H3K9me3_young, baseline_H3K9me3_old,
                          baseline_mCpG_young, baseline_mCpG_old, output_path='/private/groups/migalab/dan/fig4_blog_fold_change.svg')
plot_log_fold_changes(sub_CDR_df_merged, 
                          baseline_old_passed_CENPA, baseline_young_passed_CENPA,
                          baseline_H3K9me3_young, baseline_H3K9me3_old,
                          baseline_mCpG_young, baseline_mCpG_old,)
plot_log_fold_changes(non_island_df_merged, 
                          baseline_old_passed_CENPA, baseline_young_passed_CENPA,
                          baseline_H3K9me3_young, baseline_H3K9me3_old,
                          baseline_mCpG_young, baseline_mCpG_old)